In [65]:
import warnings
warnings.filterwarnings("ignore")

In [66]:
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import sqlite3
import mysql.connector
import pyarrow

import numpy as np
from paths import NOTEBOOKS_DIR

In [67]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")


MySQL DB Connected


In [68]:
cursor = connection.cursor()

cursor.execute("SELECT * FROM FT_HOUR_DATA")

results = cursor.fetchall()

columns = [column[0] for column in cursor.description]


df_original = pd.DataFrame(results, columns=columns)

In [69]:
df_original.sort_values(by = 'id_date', ascending = True)

,Open,High,Low,Close,adj_close,Volume,Hour,Exchange,id_exchange,id_date
0,29200,29302,29168,29268,29268,0,0,BTC-USD,193,20220521
17413,1962,1963,1944,1950,1950,0,1,ETH-USD,194,20220521
17414,1950,1961,1946,1959,1959,0,2,ETH-USD,194,20220521
17415,1958,1964,1950,1964,1964,0,3,ETH-USD,194,20220521
17416,1965,1965,1962,1963,1963,0,4,ETH-USD,194,20220521
...,...,...,...,...,...,...,...,...,...,...
17407,66815,66947,66785,66940,66940,0,19,BTC-USD,193,20240518
17408,66907,67033,66901,67006,67006,0,20,BTC-USD,193,20240518
17409,66992,67000,66852,66913,66913,0,21,BTC-USD,193,20240518
17399,67237,67372,67149,67227,67227,106489856,11,BTC-USD,193,20240518


In [70]:
df = df_original[['id_date', 'Hour', 'Close','Exchange']]

df['id_date'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
df['datetime'] = df['id_date'] + pd.to_timedelta(df['Hour'], unit='h')

df = df[['datetime', 'Close', 'Exchange']]

In [71]:
df

,datetime,Close,Exchange
0,2022-05-21 00:00:00,29268,BTC-USD
1,2022-05-21 01:00:00,29125,BTC-USD
2,2022-05-21 02:00:00,29205,BTC-USD
3,2022-05-21 03:00:00,29242,BTC-USD
4,2022-05-21 04:00:00,29227,BTC-USD
...,...,...,...
34817,2024-05-18 19:00:00,3121,ETH-USD
34818,2024-05-18 20:00:00,3124,ETH-USD
34819,2024-05-18 21:00:00,3116,ETH-USD
34820,2024-05-18 22:00:00,3119,ETH-USD


In [72]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int, 
    step_size:int
) -> list:
    
    stop_position = len(data) - 1
    
    subseq_first_idex = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []
    
    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idex, subseq_mid_idx, subseq_last_idx))
        
        subseq_first_idex += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size
        
    return indices

In [73]:
from tqdm import tqdm

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'datetime', 'Close', 'Exchange'}

    exchanges = ts_data['Exchange'].unique()
    #print(exchanges)
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for exchange in tqdm(exchanges):
        
        # keep only ts data for this `location_id`
        ts_data_one_exchange = ts_data.loc[
            ts_data.Exchange == exchange, 
            ['datetime', 'Close']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_exchange,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        hours = []
        
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_exchange.iloc[idx[0]:idx[1]]['Close'].values
            y[i] = ts_data_one_exchange.iloc[idx[1]:idx[2]]['Close'].values
            hours.append(ts_data_one_exchange.iloc[idx[1]]['datetime'])


        # numpy -> pandas
        features_one_exchange = pd.DataFrame(
            x,
            columns=[f'close_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_exchange['datetime'] = hours
        features_one_exchange['exchange'] = exchange

        # numpy -> pandas
        targets_one_exchange = pd.DataFrame(y, columns=[f'target_close_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_exchange])
        targets = pd.concat([targets, targets_one_exchange])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_close_next_hour']

In [74]:
features, targets = transform_ts_data_into_features_and_target(
    df,
    input_seq_len=24*7*1, # one week of history
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 2/2 [00:00<00:00,  3.35it/s]

features.shape=(1438, 170)
targets.shape=(1438,)


In [75]:
tabular_data = features
tabular_data['target_price_next_hour'] = targets

In [76]:
df = tabular_data

In [77]:
from typing import Tuple

def train_test_split(
    df: pd.DataFrame,
    cutoff_date: datetime,
    target_column_name: str,
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    """
    train_data = df[df.datetime < cutoff_date].reset_index(drop=True)
    test_data = df[df.datetime >= cutoff_date].reset_index(drop=True)

    X_train = train_data.drop(columns=[target_column_name])
    y_train = train_data[target_column_name]
    X_test = test_data.drop(columns=[target_column_name])
    y_test = test_data[target_column_name]

    return X_train, y_train, X_test, y_test

In [79]:
from datetime import datetime

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date=datetime(2024, 1, 1, 0, 0, 0),
    target_column_name='target_price_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')

X_train.shape=(1160, 170)
y_train.shape=(1160,)
X_test.shape=(278, 170)
y_test.shape=(278,)


In [92]:
import xgboost as xgb

In [93]:
# use only past close data
past_close_columns = [c for c in X_train.columns if c.startswith('close_')]
X_train_only_numeric = X_train[past_close_columns]

In [95]:
model = xgb.XGBRegressor()

ImportError: sklearn needs to be installed in order to use this module

In [ ]:
model.fit(X_train_only_numeric, y_train)